# Phase 5 — geometri dan replay trajectory TOR3-PAGI

Notebook CPU ini menguji geometri deterministik **sebelum** membaca track ReID
yang sudah tersimpan. Sumbernya adalah `emitted_tracks` yang terhubung ke
observasi deteksi Phase 2; anchor menggunakan BC dari `raw_bbox`.

G3 menggunakan template baru 960 × 540, dengan perimeter polygon A
`P1A → P1B → P1D → P1C` dan polygon B yang telah disetujui. Garis tetap
berupa segmen terbatas; batas polygon netral, sentuhan saja tidak cukup,
endpoint/gap ambigu, dan prediksi tidak mengesahkan crossing. Lihat
[kontrak](../docs/GEOMETRY_CONTRACT.md) dan
[provenance persetujuan](../docs/evidence/phase_5/feedback/G3_review_001.json).

**Batas interpretasi:** `GEOMETRY_TRANSITION` merupakan diagnostik software,
bukan `CountEvent`, identitas kendaraan fisik, kategori produk, atau jumlah
kendaraan. Seluruh kelas mentah tetap menjadi input diagnostik. YOLO,
tracker, dan encoder ReID tidak dimuat atau dijalankan ulang. Penghitungan
dan agregasi Phase 6 belum dimulai.

## Cara menjalankan

Di Codex, buka notebook dari checkout GitHub dan jalankan seluruh sel dengan
kernel Python. Parameter awal mengenali direktori repository saat notebook
dibuka dari root atau dari subdirektori `notebooks/`: jika `AGENTS.md` hanya
ada di direktori induk, induk menjadi `REPO_ROOT`. Ini penemuan path checkout
secara eksplisit; pada lokasi lain/Kaggle, tetapkan path aktual sendiri. Logika notebook/helper/test hanya menggunakan standard library;
IPython digunakan secara opsional untuk menampilkan gambar.

Di Kaggle, siapkan checkout repository beserta kedua helper
`notebooks/phase_5_geometry_helpers.py` dan
`notebooks/phase_5_synthetic_tests.py`. Import notebook ini, lalu ubah
`REPO_ROOT` ke checkout tersebut. GPU dan file video asli tidak dibutuhkan
untuk replay ini. Jika cache gzip berada di dataset privat Kaggle, ubah
`RECORDS_PATH` ke **file yang benar-benar tersedia**; jangan mengarang nama
dataset/path. Sesuaikan SHA-256 referensi bila sengaja memilih sumber lain.

Ubah parameter berikut sebelum **Run All**. Grid membandingkan band 0,
4.567185192815604, dan 39.945593728512996 piksel dengan konfirmasi 1 atau 2
observasi. Band 0 / konfirmasi 1 adalah **kontrol eksperimen**. Dua band lain
berasal dari median dan maksimum **tiga** excursion satu-frame yang kembali
melintasi sisi garis pada cache terukur; lihat
`docs/evidence/phase_5/20261011_G3_analysis/anchor_variation_analysis.json`.
Excursion tersebut belum terbukti sebagai jitter: gerak nyata, perubahan
komponen/kotak, dan shake masih dapat memengaruhinya. Band maksimum dipakai
sebagai stress sensitivity, sedangkan konfirmasi 2 adalah proposal
pembanding terhadap kontrol 1. Grid ini belum merupakan default yang sesuai;
hasil diagnostik tidak otomatis memilih threshold terbaik.

In [ ]:
from pathlib import Path

# Konfigurasi eksplisit. Di Kaggle, ganti REPO_ROOT ke checkout aktual.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "AGENTS.md").is_file() and (REPO_ROOT.parent / "AGENTS.md").is_file():
    REPO_ROOT = REPO_ROOT.parent  # Notebook dibuka dari notebooks/ di checkout.
RECORDS_PATH = REPO_ROOT / (
    'docs/evidence/phase_4/20261011T095924_205716Z_90cf7319/'
    'reid_on/track_records.jsonl.gz'
)
GEOMETRY_PATH = REPO_ROOT / (
    'docs/evidence/phase_5/accepted_geometry/TOR3-PAGI-G3.json'
)
PARAMETER_ANALYSIS_PATH = REPO_ROOT / (
    'docs/evidence/phase_5/20261011_G3_analysis/anchor_variation_analysis.json'
)
SOURCE_PHASE_4_RUN_ID = '20261011T095924_205716Z_90cf7319'
SOURCE_PHASE_2_RUN_ID = '20261011T052950_360560Z_ed399fb8'
FRAME_WIDTH = 1280
FRAME_HEIGHT = 720
OUTPUT_DIR = (
    Path('/kaggle/working/phase_5_geometry')
    if Path('/kaggle/working').is_dir()
    else Path('/tmp/phase_5_geometry')
)
EXPECTED_RECORDS_SHA256 = (
    '9611e2a6551e0c080692c72ac775c3ef978fb716d3900fdabbc47c79331eba89'
)
EXPECTED_VIDEO_SHA256 = (
    '606cfdd26c2d0a6892f5680d6a35d7e4d372efd76dd6bc56e2e4cbed7a74754c'
)
# Proposal eksperimen, bukan threshold produk yang telah dipilih.
# Semua band memakai HALF-WIDTH: pita netral ±neutral_band_px.
EXPERIMENT_CONFIGS = [
    {
        'experiment_id': identifier,
        'claim_type': 'PROPOSAL_EXPERIMENT_NOT_SELECTED_DEFAULT',
        'parameter_basis': basis,
        'neutral_band_px': band,
        'confirmation_observations': confirmations,
    }
    for identifier, band, confirmations, basis in (
        ('b0_c1', 0.0, 1, 'ZERO_BAND_ONE_OBSERVATION_CONTROL'),
        ('b0_c2', 0.0, 2, 'ZERO_BAND_TWO_OBSERVATION_COMPARISON_PROPOSAL'),
        ('bmedian_c1', 4.567185192815604, 1, 'MEDIAN_OF_THREE_MEASURED_ONE_FRAME_RETURN_EXCURSIONS'),
        ('bmedian_c2', 4.567185192815604, 2, 'MEDIAN_EXCURSION_PLUS_TWO_OBSERVATION_COMPARISON'),
        ('bmax_c1', 39.945593728512996, 1, 'MAXIMUM_EXCURSION_STRESS_SENSITIVITY'),
        ('bmax_c2', 39.945593728512996, 2, 'MAXIMUM_EXCURSION_STRESS_SENSITIVITY_PLUS_TWO_OBSERVATIONS'),
    )
]

# Per-video review tetap diperlukan ketika geometry/video diganti.
# None pada EXPECTED_* berarti pemeriksaan terhadap hash referensi dimatikan
# secara eksplisit; pemeriksaan hubungan observasi/video oleh helper tetap aktif.

## Lingkungan dan source code

Tahap ini hanya memeriksa lokasi file dan mengimpor helper. Isi cache track
belum dibaca. Path yang hilang menghentikan eksekusi dengan instruksi konkret.

In [ ]:
import sys
import importlib
import os
import json
import gzip
import hashlib
import platform
import re
import time
import uuid
from datetime import datetime, timezone
from collections import Counter

REPO_ROOT = Path(REPO_ROOT).resolve()
RECORDS_PATH = Path(RECORDS_PATH).resolve()
GEOMETRY_PATH = Path(GEOMETRY_PATH).resolve()
PARAMETER_ANALYSIS_PATH = Path(PARAMETER_ANALYSIS_PATH).resolve()
OUTPUT_DIR = Path(OUTPUT_DIR).resolve()
HELPER_DIR = REPO_ROOT / 'notebooks'
REQUIRED_FILES = {
    'geometry helper': HELPER_DIR / 'phase_5_geometry_helpers.py',
    'synthetic tests': HELPER_DIR / 'phase_5_synthetic_tests.py',
    'accepted geometry': GEOMETRY_PATH,
    'retained ReID track cache': RECORDS_PATH,
    'retained parameter analysis': PARAMETER_ANALYSIS_PATH,
}
missing = {name: str(path) for name, path in REQUIRED_FILES.items() if not path.is_file()}
if missing:
    raise FileNotFoundError(
        'File diperlukan belum tersedia. Perbaiki REPO_ROOT/RECORDS_PATH/'
        'GEOMETRY_PATH/PARAMETER_ANALYSIS_PATH atau sediakan checkout/cache aktual:\n'
        + json.dumps(missing, indent=2)
    )
sys.path.insert(0, str(HELPER_DIR))
import phase_5_geometry_helpers as geometry_helpers
import phase_5_synthetic_tests as synthetic_tests

# Pada rerun, gunakan code terkini sebelum merekam hash dan menjalankan tes.
importlib.invalidate_caches()
geometry_helpers = importlib.reload(geometry_helpers)
synthetic_tests = importlib.reload(synthetic_tests)

for module in (geometry_helpers, synthetic_tests):
    if Path(module.__file__).resolve().parent != HELPER_DIR.resolve():
        raise RuntimeError('Helper yang terimpor berasal dari checkout berbeda; restart kernel.')

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False) + '\n',
        encoding='utf-8',
    )

runtime = {
    'python': sys.version,
    'python_executable': sys.executable,
    'platform': platform.platform(),
    'available_cpu_count': os.cpu_count(),
    'execution_mode': 'CPU_ONLY_RETAINED_TRACK_GEOMETRY_REPLAY',
    'gpu_required': False,
    'video_decode_required': False,
    'model_import_or_inference_required': False,
}
print(json.dumps(runtime, indent=2))

## Guard: trajectory sintetis harus lulus terlebih dahulu

Hasil di bawah berasal dari tes yang benar-benar dijalankan. Bila ada
kegagalan, report tes tetap disimpan, lalu eksekusi berhenti **sebelum replay**.
Tes software ini tidak mengukur kualitas deteksi/tracking/counting.

In [ ]:
run_started_at = datetime.now(timezone.utc).isoformat()
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ') + '_' + uuid.uuid4().hex[:8]
RUN_DIR = OUTPUT_DIR / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)

source_code_sha256 = {
    'geometry_helper': sha256_file(geometry_helpers.__file__),
    'synthetic_tests': sha256_file(synthetic_tests.__file__),
}
notebook_source = HELPER_DIR / '05_geometry_replay.ipynb'
if notebook_source.is_file():
    source_code_sha256['repository_notebook_source'] = sha256_file(notebook_source)

test_result = synthetic_tests.run_tests()
write_json(RUN_DIR / 'synthetic_test_results.json', test_result)
print(json.dumps({key: test_result[key] for key in (
    'successful', 'tests_run', 'tests_passed', 'failures', 'errors', 'skipped'
)}, indent=2))
if not test_result['successful']:
    raise AssertionError(
        'Tes sintetis gagal. Replay dihentikan; bukti: '
        + str(RUN_DIR / 'synthetic_test_results.json')
    )
print('Guard lulus. Output run:', RUN_DIR)

## Geometri efektif dan identitas input

Konfigurasi ternormalisasi dipetakan ke dimensi frame yang dinyatakan.
Tidak ada snapping P2 ke vertex polygon, pemotongan garis di batas B,
perkiraan crop, atau pemetaan sisi berdasarkan tanda jarak garis saja.
Helper memvalidasi polygon sederhana, tidak tumpang tindih interiornya,
dan kecocokan dimensi/video dengan setiap record selama replay.

In [ ]:
if not EXPERIMENT_CONFIGS:
    raise ValueError('EXPERIMENT_CONFIGS harus memuat minimal satu proposal eksplisit.')
experiment_ids = set()
for config in EXPERIMENT_CONFIGS:
    identifier = config['experiment_id']
    if not isinstance(identifier, str) or re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*', identifier) is None:
        raise ValueError('experiment_id harus berupa nama file sederhana tanpa slash.')
    if identifier in experiment_ids:
        raise ValueError('experiment_id tidak boleh berulang dalam satu run.')
    experiment_ids.add(identifier)
    if not config.get('claim_type'):
        raise ValueError('Setiap konfigurasi membutuhkan claim_type eksperimen eksplisit.')
    geometry_helpers.GeometryStateMachine(
        geometry_helpers.Geometry.from_draft(GEOMETRY_PATH, FRAME_WIDTH, FRAME_HEIGHT),
        neutral_band_px=config['neutral_band_px'],
        confirmation_observations=config['confirmation_observations'],
    )

geometry = geometry_helpers.Geometry.from_draft(GEOMETRY_PATH, FRAME_WIDTH, FRAME_HEIGHT)
input_sha256 = {
    'retained_records': sha256_file(RECORDS_PATH),
    'accepted_geometry': sha256_file(GEOMETRY_PATH),
    'retained_parameter_analysis': sha256_file(PARAMETER_ANALYSIS_PATH),
}
if EXPECTED_RECORDS_SHA256 is not None and input_sha256['retained_records'] != EXPECTED_RECORDS_SHA256:
    raise ValueError('SHA-256 cache berbeda dari sumber yang dikonfigurasi. Jangan lanjut tanpa review input.')
if EXPECTED_VIDEO_SHA256 is not None and geometry.video_sha256 != EXPECTED_VIDEO_SHA256:
    raise ValueError('Identitas video pada geometri berbeda dari sumber yang dikonfigurasi.')
write_json(RUN_DIR / 'effective_geometry.json', geometry.as_dict())
print(json.dumps({'input_sha256': input_sha256, 'effective_geometry': geometry.as_dict()}, indent=2))

## Replay cache dengan parameter eksperimen eksplisit

Setiap konfigurasi membaca cache yang sama dan menggunakan geometri yang
sama. Observasi hilang tidak diinterpolasi. Prediksi/lifecycle bbox tidak
menjadi pengganti pengukuran; kelas mentah tidak dipetakan ke kategori produk.

Report menyimpan identitas input, source code, konfigurasi, timestamp run,
dan durasi eksekusi yang benar-benar diukur. Nilai diagnostik software tidak
dapat dipakai sebagai jumlah kendaraan atau bukti peningkatan performa.
`parent_run_id` di setiap record adalah run deteksi **Phase 2**, bukan run
tracker/ReID. Report memisahkan referensi run **Phase 4** yang dikonfigurasi
dan parent **Phase 2** yang benar-benar diverifikasi dari cache.


In [ ]:
experiment_results = []
for config in EXPERIMENT_CONFIGS:
    started_at = datetime.now(timezone.utc).isoformat()
    started_clock = time.perf_counter()
    replay = geometry_helpers.replay_cached_records(
        RECORDS_PATH,
        geometry,
        neutral_band_px=config['neutral_band_px'],
        confirmation_observations=config['confirmation_observations'],
    )
    elapsed_seconds = time.perf_counter() - started_clock
    report = replay['report']
    if report['source_records_file_sha256'] != input_sha256['retained_records']:
        raise RuntimeError('Cache berubah setelah pemeriksaan hash; run dihentikan.')
    if report['video_sha256'] != geometry.video_sha256:
        raise RuntimeError('Identitas video report tidak cocok dengan geometri.')

    if report['source_parent_run_id'] != SOURCE_PHASE_2_RUN_ID or report['source_variant'] != 'reid_on':
        raise RuntimeError('Sumber parent Phase2 / variant tidak cocok dengan baseline yang dikonfigurasi.')

    identifier = config['experiment_id']
    diagnostics_path = RUN_DIR / (identifier + '_diagnostics.jsonl.gz')
    # mtime=0 menjaga kompresi deterministik untuk isi diagnostik yang identik.
    with diagnostics_path.open('wb') as output_file:
        with gzip.GzipFile(filename='', mode='wb', fileobj=output_file, mtime=0) as compressed:
            for diagnostic in replay['diagnostics']:
                compressed.write((json.dumps(
                    diagnostic, ensure_ascii=False, allow_nan=False, sort_keys=True
                ) + '\n').encode('utf-8'))

    transition_directions = dict(Counter(
        diagnostic['direction'] for diagnostic in replay['diagnostics']
        if diagnostic['type'] == 'GEOMETRY_TRANSITION'
    ))
    execution_report = {
        'run_id': run_id,
        'source_phase_4_run_id': SOURCE_PHASE_4_RUN_ID,
        'source_parent_phase_2_run_id': report['source_parent_run_id'],
        'execution_started_at_utc': started_at,
        'execution_completed_at_utc': datetime.now(timezone.utc).isoformat(),
        'measured_replay_elapsed_seconds': elapsed_seconds,
        'runtime': runtime,
        'experiment_config': dict(config),
        'source_code_sha256': dict(source_code_sha256),
        'input_sha256': dict(input_sha256),
        'synthetic_tests_successful': test_result['successful'],
        'synthetic_test_results_sha256': sha256_file(RUN_DIR / 'synthetic_test_results.json'),
        'software_report': report,
        'software_transition_direction_diagnostics': transition_directions,
        'diagnostics_file': diagnostics_path.name,
        'diagnostics_file_sha256': sha256_file(diagnostics_path),
        'selected_suitable_product_parameters': None,
        'selected_parameter_status': 'OPEN_REQUIRES_TEMPORAL_REVIEW',
        'phase_6_started': False,
        'limitations': [
            'Software track identity does not establish unique physical vehicles.',
            'All raw classes are diagnostic inputs; no accepted product mapping.',
            'Repeated geometric transitions are not accepted repeated count events.',
            'No source-video decode or complete continuous visual review is performed here.',
            'Index/FPS timestamps are estimates, not verified source PTS.',
            'No ground truth or real-world accuracy evidence is established.',
        ],
    }
    report_path = RUN_DIR / (identifier + '_report.json')
    write_json(report_path, execution_report)
    experiment_results.append(execution_report)
    print(identifier, 'selesai;', report['frames_processed'], 'record frame tervalidasi;',
          round(elapsed_seconds, 3), 'detik replay CPU')
    del replay

for name, path in (
    ('geometry_helper', geometry_helpers.__file__),
    ('synthetic_tests', synthetic_tests.__file__),
):
    if sha256_file(path) != source_code_sha256[name]:
        raise RuntimeError('Source code berubah selama run: ' + name)
if sha256_file(GEOMETRY_PATH) != input_sha256['accepted_geometry']:
    raise RuntimeError('Geometri berubah selama run; hasil memerlukan investigasi.')
if sha256_file(PARAMETER_ANALYSIS_PATH) != input_sha256['retained_parameter_analysis']:
    raise RuntimeError('Bukti asal parameter berubah selama run.')

artifact_paths = sorted(path for path in RUN_DIR.iterdir() if path.is_file())
manifest = {
    'run_id': run_id,
    'run_started_at_utc': run_started_at,
    'run_completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'claim_type': 'EXPERIMENT_RESULT_SOFTWARE_GEOMETRY_REPLAY_ONLY',
    'source_code_sha256': source_code_sha256,
    'input_sha256': input_sha256,
    'experiment_configs': EXPERIMENT_CONFIGS,
    'suitable_product_parameter_decision': 'OPEN',
    'artifacts': [
        {'path': path.name, 'size_bytes': path.stat().st_size, 'sha256': sha256_file(path)}
        for path in artifact_paths
    ],
}
write_json(RUN_DIR / 'execution_manifest.json', manifest)

## Ringkasan diagnostik dan review berikutnya

Kolom A→B/B→A berikut adalah banyaknya **diagnostik transition geometri**,
termasuk seluruh kelas mentah; kolom ini bukan traffic count. Perbedaan
antar konfigurasi menjelaskan perilaku aturan dan perlu ditinjau pada
trajectory/frame terkait. Lebih sedikit atau lebih banyak diagnostik tidak
otomatis berarti lebih baik.

Simpan JSON report, gzip diagnostik, hasil tes, dan manifest. Tinjau sumber
frame/clip yang benar-benar tersedia sebelum memilih parameter. Kebutuhan
deduplikasi identitas, kebijakan kategori, dan genuine repeat/U-turn tetap
OPEN untuk Phase 6; notebook ini tidak mengambil keputusan tersebut.

In [ ]:
columns = ('experiment', 'band_halfwidth_px', 'confirm_obs', 'A_TO_B', 'B_TO_A',
           'ambiguous_gap', 'ambiguous_endpoint')
rows = []
for result in experiment_results:
    counts = result['software_report']['software_diagnostic_counts']
    directions = result['software_transition_direction_diagnostics']
    config = result['experiment_config']
    rows.append((config['experiment_id'], config['neutral_band_px'],
                 config['confirmation_observations'], directions.get('A_TO_B', 0),
                 directions.get('B_TO_A', 0), counts.get('AMBIGUOUS_GAP', 0),
                 counts.get('AMBIGUOUS_ENDPOINT', 0)))
print('DIAGNOSTIK SOFTWARE — BUKAN JUMLAH KENDARAAN')
print(' | '.join(columns))
for row in rows:
    print(' | '.join(str(value) for value in row))
print('Keputusan threshold yang sesuai: OPEN; tidak dipilih otomatis.')
print('Bukti run:', RUN_DIR)
print('Manifest:', RUN_DIR / 'execution_manifest.json')

## Opsional: tampilkan frame sumber yang tersimpan

Frame ini berasal dari bukti Phase 2, bukan hasil decode baru. Penampilannya
tidak mengeksekusi review temporal atau mengesahkan transition. Jika ingin
mengevaluasi kasus tertentu, gunakan frame/clip beserta trajectory yang
memuat origin, crossing, dan konfirmasi; catat cakupan dan batas review.

In [ ]:
source_frame_path = REPO_ROOT / (
    'docs/evidence/phase_2/20261011T052950_360560Z_ed399fb8/'
    'samples/frame_000005140_raw.png'
)
if source_frame_path.is_file():
    try:
        from IPython.display import Image, display
    except ImportError:
        print('IPython tidak tersedia. Buka frame secara manual:', source_frame_path)
    else:
        print('Frame sumber 5140; tidak ada temuan visual otomatis.')
        display(Image(filename=str(source_frame_path)))
else:
    print('Frame sumber tidak tersedia di checkout ini; review visual belum dilakukan.')